In [ ]:
#Summary Statistics:
!pip install pandas

In [33]:
import pandas as pd
df_KF= pd.read_csv("GeneExpression/Datasets/set_16/k_vs_F.deseq2.results.tsv", sep = "\t")
df_KL= pd.read_csv("GeneExpression/Datasets/set_16/k_vs_L.deseq2.results.tsv", sep = "\t")

In [ ]:
df_KF.head()

In [ ]:
df_KL.head()

In [20]:
padj_cutoff = 0.05 # Adjusted p-value threshold (significance requires padj < this value)
logFC_up = 1  # Threshold for upregulation: log2FoldChange > 1
logFC_down = -1

In [21]:
KF_up = df_KF[(df_KF["padj"] < padj_cutoff) & (df_KF["log2FoldChange"] > logFC_up)] #Select significantly upregulated genes (padj < 0.05 AND log2FC > 1)
KF_down = df_KF[(df_KF["padj"] < padj_cutoff) & (df_KF["log2FoldChange"] < logFC_down)] #Select significantly downregulated genes (padj < 0.05 AND log2FC < -1)


In [22]:
KL_up = df_KL[(df_KL["padj"] < padj_cutoff) & (df_KL["log2FoldChange"] > logFC_up)]
KL_down = df_KL[(df_KL["padj"] < padj_cutoff) & (df_KL["log2FoldChange"] < logFC_down)]


In [ ]:
summary = pd.DataFrame({
    "Comparison": ["K_vs_F", "K_vs_L"],
    "Upregulated": [len(KF_up), len(KL_up)],
    "Downregulated": [len(KF_down), len(KL_down)],
    "Total_DEGs": [len(KF_up)+len(KF_down), len(KL_up)+len(KL_down)]
})

summary


In [ ]:
#Summary of p-values and log fold changes across all genes for each comparison.
def summary_stats(df):
    print("\nSummary of p-values:")
    print("Mean:", df["pvalue"].mean())
    print("Median:", df["pvalue"].median())
    print("Min:", df["pvalue"].min())
    print("Max:", df["pvalue"].max())

    print("\nSummary of log2FoldChange:")
    print("Mean:", df["log2FoldChange"].mean())
    print("Median:", df["log2FoldChange"].median())
    print("Min:", df["log2FoldChange"].min())
    print("Max:", df["log2FoldChange"].max())


print("\n=== K vs F Summary ===")
summary_stats(df_KF)

print("\n=== K vs L Summary ===")
summary_stats(df_KL)

In [ ]:
!pip install matplotlib 

In [ ]:
import seaborn as sns # Import seaborn for  scatter plotting
import numpy as np # Import numpy for numerical operations
import matplotlib.pyplot as plt # Import matplotlib for plotting functions

def Volcano_plot(df, title, padj_cutoff=0.05, logFC_cutoff=1):
    df = df.copy()
    df["-log10padj"] = -np.log10(df["padj"])   # Calculation of Significance on the Y-axis
    df["status"] = "Not Sig" #gene classification
    df.loc[(df["padj"]<padj_cutoff) & (df["log2FoldChange"]>logFC_cutoff),"status"] = "Up"
    #Genes with adjusted p-value < 0.05 and log2FC > 1 were classified as Up-regulated,while those with log2FC < −1 were defined as Down-regulated.
    df.loc[(df["padj"]<padj_cutoff) & (df["log2FoldChange"]<-logFC_cutoff),"status"] = "Down"
    colors = {"Up":"red", "Down":"blue", "Not Sig":"gray"}
    plt.figure(figsize=(6,5))
    
    for group in ["Not Sig","Up","Down"]:
        sub = df[df["status"]==group]
        plt.scatter(sub["log2FoldChange"],sub["-log10padj"],
                    color=colors[group],s=15,label=group)
        
    plt.axvline(logFC_cutoff, color="black", linestyle="--")
    plt.axvline(-logFC_cutoff, color="black", linestyle="--")
    plt.axhline(-np.log10(padj_cutoff), color="black", linestyle="--")

    plt.xlabel("log2FoldChange")
    plt.ylabel("-log10(padj)")
    plt.title(title)
    plt.legend(title="Differential Expression")
    plt.show()

Volcano_plot(df_KF, "Volcano_plot - KF")
Volcano_plot(df_KL, "Volcano_plot - KL")



In [ ]:

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

def MA_plot(df, title, padj_cutoff=0.05, logFC_cutoff=1):
    df = df.copy()
    df["logBaseMean"] = np.log10(df["baseMean"] + 1)  # Convert mean expression to log10 scale for MA plot  
    #gene classification
    df["status"] = "Not Sig"
    df.loc[(df["padj"]<padj_cutoff) & (df["log2FoldChange"]>logFC_cutoff),"status"] = "Up"
    df.loc[(df["padj"]<padj_cutoff) & (df["log2FoldChange"]<-logFC_cutoff),"status"] = "Down"
    colors = {"Up":"red", "Down":"blue", "Not Sig":"gray"}
    plt.figure(figsize=(6,5))
    
    for group in ["Not Sig","Up","Down"]:
        sub = df[df["status"]==group] #Filter dataframe to only include genes of this category
        plt.scatter(sub["logBaseMean"], sub["log2FoldChange"],
                    s=15, color=colors[group], label=group)
        
    plt.axhline(0, color="black", linestyle="--") #Horizontal reference line showing no fold-change boundary
    plt.xlabel("log10(baseMean)")
    plt.ylabel("log2FoldChange")
    plt.title(title)
    plt.legend(title="Expression Change")
    plt.show()


MA_plot(df_KF, "MA Plot - KF")
MA_plot(df_KL, "MA Plot - KL")

In [ ]:
#Histogram of p-values to assess the distribution of statistical significance.
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

def pval_hist(df, title):
    plt.figure(figsize=(6,5))
    plt.hist(df["pvalue"], bins=50, color="skyblue", edgecolor="black")
    plt.xlabel("p-value")
    plt.ylabel("Frequency")
    plt.title(title)
    plt.show()

pval_hist(df_KF, "P-value Histogram - KF")
pval_hist(df_KL, "P-value Histogram - KL")


In [ ]:
#Heatmap of the top differentially expressed genes to illustrate gene expression patterns across the conditions.

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def heatmap_DEG_logFC(df, title, topN=30):
   #  Select significantly differentially expressed genes
    df_sig = df[df["padj"] < 0.05].copy() ##  Select significantly differentially expressed genes
    df_sig = df_sig.sort_values("log2FoldChange", key=abs, ascending=False).head(topN) #sort genes by absolute fold change magnitude,keep Top N strongest DEGs
    heatmap_data = df_sig[["log2FoldChange"]] #Select only "log2FoldChange" column for heatmap
    heatmap_data.index = df_sig["gene_id"] #Set "gene_id" as row index to label genes in heatmap
    plt.figure(figsize=(10,14))
    sns.heatmap(heatmap_data, cmap="bwr", center=0, annot=True, fmt=".2f") #Set the image size, draw a heatmap, with red indicating an increase and blue indicating a decrease, with 0 as the color center, and keep the precision to two decimal places.
    plt.title(f"{title} — Top {topN} DEGs")
    plt.xlabel("log2FoldChange (Expression Trend)")
    plt.ylabel("Genes")
    plt.savefig(f"/Users/nicole/Desktop/{title}.png", dpi=400)
    plt.show()

heatmap_DEG_logFC(df_KF,"KF Heatmap")
heatmap_DEG_logFC(df_KL,"KL Heatmap")



In [ ]:
#A table or list of significantly upregulated and downregulated genes with their corresponding fold changes, p-values, and adjusted p-values.
def DEG_table(df, padj_cutoff=0.05, logFC_cutoff=1):
    df = df.copy()
    up = df[(df["padj"] < padj_cutoff) & (df["log2FoldChange"] > logFC_cutoff)]# Significantly upregulated genes
    down = df[(df["padj"] < padj_cutoff) & (df["log2FoldChange"] < -logFC_cutoff)]# Significantly downregulated genes
    print(f" Significantly Upregulated genes: {len(up)}")
    print(f" Significantly Downregulated genes: {len(down)}\n")
    up_table = up[["gene_id","log2FoldChange", "pvalue", "padj"]]# Extracts columns of interest from significantly upregulated genes.
    down_table = down[["gene_id","log2FoldChange", "pvalue", "padj"]]

    return up_table, down_table


In [60]:
up_KF, down_KF = DEG_table(df_KF)
up_KL, down_KL = DEG_table(df_KL)

up_KF.head(), down_KF.head()


In [98]:
# Additional Analyses:
Based solely on visual output, the experimental condition triggers non-random gene expression shifts, resulting in clear separation of regulated genes.
Clustering patterns indicate that DEGs are functionally connected rather than independent responders.
The system likely undergoes pathway-level regulation, where groups of genes are switched ON/OFF to adapt to the condition.